# Reliability & Calibration Analysis — S3D / KTH Runs

Scans every `s3d_v2_*` folder inside `RESULTS_ROOT`, loads `best_model.pth`,
runs inference on the **test split** using the exact same `KTHVideoDataset`
and `S3D_Weights.KINETICS400_V1` transforms used during training, and produces:
- Reliability diagram with accuracy + clip_len annotated
- Per-bin comparison (claimed confidence vs actual accuracy)
- ECE (Expected Calibration Error)
- Saves `calibration_curve.png` into each run folder
- Cross-run summary table and comparison chart

## 0 · Configuration — **edit these**

In [1]:
# ── paths ─────────────────────────────────────────────────────────────────────
RESULTS_ROOT = "./S3D_runs"      # root that contains s3d_v2_* run folders
SPLIT_DIR    = "./kth_split"      # partitioned dataset root (train/validation/test)
EVAL_SPLIT   = "test"             # which split to evaluate on

# ── data ──────────────────────────────────────────────────────────────────────
BATCH_SIZE   = 8                  # match training batch size
N_BINS       = 10                 # calibration bins

# ── which runs to process (None = all s3d_v2_* folders found) ─────────────────
RUN_FILTER   = None               # e.g. ["s3d_v2_20260518_121015"]

## 1 · Imports

In [2]:
import os
import json
import random
from pathlib import Path

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.video import s3d, S3D_Weights
from torchvision.datasets.folder import make_dataset
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# KTH class names — fixed, same order as training
CLASS_NAMES = ['boxing', 'handclapping', 'handwaving', 'jogging', 'running', 'walking']

Using device: cpu


## 2 · KTHVideoDataset (identical to training notebook)

In [3]:
def _find_classes(dir):
    classes = [d.name for d in os.scandir(dir) if d.is_dir()]
    classes.sort()
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    return classes, class_to_idx

def get_samples(root, extensions=(".mp4", ".avi")):
    _, class_to_idx = _find_classes(root)
    return make_dataset(root, class_to_idx, extensions=extensions)


class KTHVideoDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None, clip_len=64, train=False):
        super().__init__()
        all_samples = get_samples(root)
        self.clip_len  = clip_len
        self.transform = transform
        self.train     = train   # False -> deterministic center/start crop for eval

        self.samples = []
        for path, target in all_samples:
            cap = cv2.VideoCapture(path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            if total_frames >= 1:
                self.samples.append((path, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]

        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(torch.from_numpy(frame).permute(2, 0, 1))
        cap.release()

        total_frames = len(frames)
        if total_frames == 0:
            raise RuntimeError(f"No frames could be read from: {path}")

        if total_frames >= self.clip_len:
            # deterministic center crop for eval (same logic as training notebook)
            start   = (total_frames - self.clip_len) // 2
            indices = [start + i for i in range(self.clip_len)]
        else:
            # loop clip cyclically (same logic as training notebook)
            indices = [i % total_frames for i in range(self.clip_len)]

        video = torch.stack([frames[i] for i in indices], 0).float() / 255.0

        if self.transform:
            video = self.transform(video)

        return video, target

## 3 · Model builder (matches training notebook exactly)

In [4]:
def build_s3d(num_classes: int) -> nn.Module:
    """Rebuild the exact same S3D skeleton used in setup_model()."""
    weights = S3D_Weights.KINETICS400_V1
    model   = s3d(weights=weights)
    # Replace classifier head exactly as done during training
    in_channels = model.classifier[1].in_channels
    model.classifier[1] = nn.Conv3d(in_channels, num_classes, kernel_size=1)
    return model

# Same transforms used in training (loaded once, shared across all runs)
PREPROCESS = S3D_Weights.KINETICS400_V1.transforms()

## 4 · Calibration core functions

In [5]:
def collect_predictions(model, dataloader, device):
    """Return (all_confidences, all_correct) lists over the full dataloader."""
    model.eval()
    all_confidences, all_correct = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            probs        = F.softmax(model(inputs), dim=1)
            confidences  = probs.max(dim=1).values
            predictions  = probs.argmax(dim=1)
            correct      = (predictions == labels).float()

            all_confidences.extend(confidences.cpu().tolist())
            all_correct.extend(correct.cpu().tolist())

    return all_confidences, all_correct


def compute_bins(all_confidences, all_correct, n_bins=10):
    bin_edges = [i / n_bins for i in range(n_bins + 1)]
    bin_mean_conf, bin_accuracy, bin_labels, bin_counts = [], [], [], []

    for i in range(n_bins):
        low, high = bin_edges[i], bin_edges[i + 1]
        in_bin = [(c, cor)
                  for c, cor in zip(all_confidences, all_correct)
                  if low <= c < high]
        if not in_bin:
            continue
        confs, cors = zip(*in_bin)
        bin_mean_conf.append(sum(confs) / len(confs))
        bin_accuracy.append(sum(cors)  / len(cors))
        bin_labels.append(f"{low:.1f}-{high:.1f}")
        bin_counts.append(len(in_bin))

    return bin_mean_conf, bin_accuracy, bin_labels, bin_counts


def compute_ece(all_confidences, all_correct, n_bins=10):
    n         = len(all_confidences)
    bin_edges = [i / n_bins for i in range(n_bins + 1)]
    ece = 0.0
    for i in range(n_bins):
        low, high = bin_edges[i], bin_edges[i + 1]
        in_bin = [(c, cor)
                  for c, cor in zip(all_confidences, all_correct)
                  if low <= c < high]
        if not in_bin:
            continue
        confs, cors = zip(*in_bin)
        ece += (len(in_bin) / n) * abs(sum(cors)/len(cors) - sum(confs)/len(confs))
    return ece


def plot_calibration_curve(bin_mean_conf, bin_accuracy, bin_labels, bin_counts,
                            ece, run_name, overall_acc, clip_len, save_dir, n_bins=10):
    clip_str = f"clip_len={clip_len}" if clip_len is not None else "clip_len=N/A"

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        f"Reliability — {run_name}\n"
        f"Accuracy = {overall_acc:.4f}  |  {clip_str}  |  ECE = {ece:.4f}",
        fontsize=12, fontweight="bold"
    )

    # --- left: reliability diagram ---
    ax1.bar(bin_mean_conf, bin_accuracy, width=1/n_bins*0.9,
            alpha=0.8, color="steelblue", label="this model")
    ax1.plot([0, 1], [0, 1], linestyle="--", color="black", label="perfect calibration")
    stats_text = f"Accuracy : {overall_acc:.4f}\n{clip_str}\nECE      : {ece:.4f}"
    ax1.text(0.03, 0.97, stats_text, transform=ax1.transAxes,
             fontsize=8, verticalalignment="top",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow",
                       edgecolor="gray", alpha=0.85))
    ax1.set_xlabel("What the model claimed (avg confidence)")
    ax1.set_ylabel("What actually happened (fraction correct)")
    ax1.set_title("Reliability diagram")
    ax1.set_xlim(0.4, 1.0)
    ax1.set_ylim(0.4, 1.0)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- right: per-bin grouped bar chart ---
    bar_width = 0.35
    x         = list(range(len(bin_labels)))
    ax2.bar([i - bar_width/2 for i in x], bin_mean_conf, width=bar_width,
            color="orange", alpha=0.9, label="claimed (avg confidence)")
    ax2.bar([i + bar_width/2 for i in x], bin_accuracy,  width=bar_width,
            color="steelblue", alpha=0.9, label="actual (fraction correct)")

    for xi, cnt in zip(x, bin_counts):
        top = max(bin_mean_conf[xi], bin_accuracy[xi])
        ax2.text(xi, top + 0.02, f"n={cnt}",
                 ha="center", va="bottom", fontsize=7, color="gray")

    ax2.set_xticks(x)
    ax2.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax2.set_xlabel("Confidence bin")
    ax2.set_ylabel("Probability")
    ax2.set_title("Per-bin comparison: claimed vs actual")
    ax2.set_ylim(0, 1.0)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    out_path = os.path.join(save_dir, "calibration_curve.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved -> {out_path}")
    return out_path

## 5 · Discover runs

In [6]:
all_run_dirs = sorted(
    p for p in Path(RESULTS_ROOT).iterdir()
    if p.is_dir() and p.name.startswith("s3d_v2_")
)

if RUN_FILTER:
    all_run_dirs = [p for p in all_run_dirs if p.name in RUN_FILTER]

print(f"Found {len(all_run_dirs)} run(s):")
for r in all_run_dirs:
    ckpt = (r / "best_model.pth").exists()
    hp   = (r / "hyperparams.json").exists()
    print(f"  {r.name}  | checkpoint={'OK' if ckpt else 'MISSING'}  hyperparams={'OK' if hp else 'MISSING'}")

Found 3 run(s):
  s3d_v2_20260604_165754  | checkpoint=OK  hyperparams=OK
  s3d_v2_20260604_170136  | checkpoint=OK  hyperparams=OK
  s3d_v2_20260604_171418  | checkpoint=OK  hyperparams=OK


## 6 · Run calibration analysis over all runs

In [7]:
summary = []

for run_dir in all_run_dirs:
    run_name  = run_dir.name
    ckpt_path = run_dir / "best_model.pth"
    hp_path   = run_dir / "hyperparams.json"

    print(f"\n{'='*60}")
    print(f"  Run: {run_name}")
    print(f"{'='*60}")

    if not ckpt_path.exists():
        print(f"  [SKIP] No checkpoint at {ckpt_path}")
        continue

    # load hyperparams
    hyperparams = {}
    if hp_path.exists():
        with open(hp_path) as f:
            hyperparams = json.load(f)
        print(f"  Hyperparams: {hyperparams}")

    # CLIP_LEN is the exact key saved by the training notebook
    clip_len = hyperparams.get("CLIP_LEN")
    print(f"  clip_len          : {clip_len if clip_len is not None else 'not found in hyperparams'}")

    # build dataset with this run's clip_len
    test_dir     = os.path.join(SPLIT_DIR, EVAL_SPLIT)
    dataset_test = KTHVideoDataset(
        root      = test_dir,
        transform = PREPROCESS,
        clip_len  = clip_len if clip_len is not None else 64,
        train     = False
    )
    test_loader = DataLoader(
        dataset_test, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=4, pin_memory=True
    )
    print(f"  Test samples      : {len(dataset_test)}")

    # build model + load raw state_dict (training notebook saves model.state_dict() directly)
    try:
        model = build_s3d(num_classes=len(CLASS_NAMES)).to(device)
    except Exception as e:
        print(f"  [SKIP] Could not build model: {e}")
        continue

    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    elif isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)
    print(f"  Loaded weights from {ckpt_path.name}")

    # inference
    print(f"  Running inference on '{EVAL_SPLIT}' split ...")
    all_confidences, all_correct = collect_predictions(model, test_loader, device)
    print(f"  Samples evaluated : {len(all_confidences)}")

    # metrics
    ece         = compute_ece(all_confidences, all_correct, n_bins=N_BINS)
    overall_acc = sum(all_correct) / len(all_correct)
    avg_conf    = sum(all_confidences) / len(all_confidences)
    print(f"  Overall accuracy  : {overall_acc:.4f}")
    print(f"  Average confidence: {avg_conf:.4f}")
    print(f"  ECE               : {ece:.4f}")

    # plot
    bmc, ba, bl, bc = compute_bins(all_confidences, all_correct, n_bins=N_BINS)
    plot_calibration_curve(
        bmc, ba, bl, bc,
        ece, run_name, overall_acc, clip_len,
        save_dir=str(run_dir), n_bins=N_BINS
    )

    summary.append({
        "run"          : run_name,
        "clip_len"     : clip_len,
        "accuracy"     : overall_acc,
        "avg_conf"     : avg_conf,
        "ECE"          : ece,
        "overconfident": avg_conf > overall_acc,
    })

    del model
    torch.cuda.empty_cache()


  Run: s3d_v2_20260604_165754
  Hyperparams: {'SEED': 0, 'TEST_SPLIT': 0.2, 'VAL_SPLIT': 0.2, 'BATCH_SIZE': 8, 'NR_EPOCHS': 30, 'CLIP_LEN': 32, 'OPTIMIZER': 'SGD', 'LEARNING_RATE': 0.01, 'MOMENTUM': 0.9, 'WEIGHT_DECAY': 1e-05, 'LR_STEP_SIZE': 10, 'LR_GAMMA': 0.1, 'FREEZE': True, 'UNFREEZE_EPOCH': 15, 'MODEL': 'S3D (Kinetics400 Pre-trained)'}
  clip_len          : 32
  Test samples      : 863
  Loaded weights from best_model.pth
  Running inference on 'test' split ...


c:\Users\User\anaconda3\envs\camera_EvacAware\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: DataLoader worker (pid(s) 25228, 20044, 23568, 596) exited unexpectedly

## 7 · Cross-run summary table

In [ ]:
if summary:
    print(f"\n{'Run':<35} {'clip_len':>10} {'Accuracy':>10} {'Avg conf':>10} {'ECE':>8} {'Over-conf?':>12}")
    print("-" * 90)
    for row in sorted(summary, key=lambda r: r["ECE"]):
        flag = "yes (!)" if row["overconfident"] else "no"
        cl   = str(row["clip_len"]) if row["clip_len"] is not None else "N/A"
        print(f"{row['run']:<35} {cl:>10} {row['accuracy']:>10.4f} "
              f"{row['avg_conf']:>10.4f} {row['ECE']:>8.4f} {flag:>12}")
else:
    print("No runs were processed successfully.")

## 8 · ECE vs Accuracy comparison chart across runs

In [ ]:
if len(summary) > 1:
    summary_sorted = sorted(summary, key=lambda r: r["ECE"])
    tick_labels = [
        f"{r['run']}\nclip={r['clip_len'] if r['clip_len'] is not None else 'N/A'}"
        for r in summary_sorted
    ]
    ece_vals = [r["ECE"]      for r in summary_sorted]
    acc_vals = [r["accuracy"] for r in summary_sorted]

    x = np.arange(len(tick_labels))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(tick_labels) * 2.5), 5))
    ax.bar(x - w/2, ece_vals, width=w, color="tomato",    alpha=0.85, label="ECE (lower = better)")
    ax.bar(x + w/2, acc_vals, width=w, color="steelblue", alpha=0.85, label="Accuracy")
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=30, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("ECE vs Accuracy across runs (sorted by ECE)")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    out = os.path.join(RESULTS_ROOT, "runs_calibration_comparison.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out}")